# MERRA-2 Air Pollution Data Integration

This notebook:
1. Reads all 182 MERRA-2 monthly aerosol NC4 files (2010–2024)
2. Builds a lookup table indexed by (year-month, lat, lon)
3. For each observation in `final2.csv`, finds the nearest grid cell and matching month
4. Adds 8 aerosol pollution variables to the dataset and saves the result

In [1]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import netCDF4 as nc

warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

Libraries loaded successfully.


## Step 1: Read all MERRA-2 NC4 files and build a pollution lookup table

In [2]:
# Path to air pollution NC4 files
nc4_dir = 'air pollution'
nc4_files = sorted(glob.glob(os.path.join(nc4_dir, 'MERRA2_*.nc4.nc4')))
print(f'Found {len(nc4_files)} NC4 files')

# Aerosol variables to extract
POLLUTION_VARS = [
    'BCSMASS',    # Black Carbon Surface Mass Concentration (kg/m³)
    'DUEXTT25',   # Dust Extinction AOT [550 nm] - PM 2.5
    'DUSMASS',    # Dust Surface Mass Concentration (kg/m³)
    'OCSMASS',    # Organic Carbon Surface Mass Concentration (kg/m³)
    'SO2SMASS',   # SO2 Surface Mass Concentration (kg/m³)
    'SO4SMASS',   # SO4 Surface Mass Concentration (kg/m³)
    'SSSMASS',    # Sea Salt Surface Mass Concentration (kg/m³)
    'TOTEXTTAU',  # Total Aerosol Extinction AOT [550 nm]
    'TOTSCATAU',  # Total Aerosol Scattering AOT [550 nm]
]

# Read all NC4 files and store in a dictionary keyed by 'YYYY-MM'
# Each entry holds: { 'lat': array, 'lon': array, var_name: 2D array (lat x lon) }
pollution_lookup = {}

for filepath in nc4_files:
    # Extract year-month from filename (e.g., '201001' from the filename)
    basename = os.path.basename(filepath)
    # Pattern: MERRA2_XXX.tavgM_2d_aer_Nx.YYYYMM.nc4.nc4
    parts = basename.split('.')
    yyyymm = parts[2]  # e.g., '201001'
    year_month = f'{yyyymm[:4]}-{yyyymm[4:6]}'  # e.g., '2010-01'
    
    ds = nc.Dataset(filepath)
    
    lat = ds.variables['lat'][:].data
    lon = ds.variables['lon'][:].data
    
    entry = {'lat': lat, 'lon': lon}
    for var in POLLUTION_VARS:
        if var in ds.variables:
            # Shape is (1, lat, lon) — squeeze time dimension
            entry[var] = ds.variables[var][0, :, :].data
        else:
            entry[var] = np.full((len(lat), len(lon)), np.nan)
    
    # Handle duplicate months (e.g., MERRA2_401 files) — keep the latest version
    pollution_lookup[year_month] = entry
    ds.close()

print(f'\nLoaded pollution data for {len(pollution_lookup)} unique months')
print(f'Date range: {min(pollution_lookup.keys())} to {max(pollution_lookup.keys())}')
print(f'Grid: {len(lat)} lat points × {len(lon)} lon points')
print(f'Lat range: {lat.min():.2f} to {lat.max():.2f}')
print(f'Lon range: {lon.min():.3f} to {lon.max():.3f}')
print(f'Variables: {POLLUTION_VARS}')

Found 180 NC4 files

Loaded pollution data for 180 unique months
Date range: 2010-01 to 2024-12
Grid: 9 lat points × 5 lon points
Lat range: 6.00 to 10.00
Lon range: 79.375 to 81.875
Variables: ['BCSMASS', 'DUEXTT25', 'DUSMASS', 'OCSMASS', 'SO2SMASS', 'SO4SMASS', 'SSSMASS', 'TOTEXTTAU', 'TOTSCATAU']


## Step 2: Load final2.csv

In [3]:
df = pd.read_csv('final2.csv')
print(f'Loaded final2.csv: {len(df):,} rows × {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')
df.head(3)

Loaded final2.csv: 1,999,208 rows × 12 columns
Columns: ['index', 'verbatimScientificName', 'stateProvince', 'individualCount', 'decimalLatitude', 'decimalLongitude', 'eventDate', 'avg_rad', 'cf_cvg', 'NDVI_raw', 'LandCover_Class', 'elevation_meters']


,index,verbatimScientificName,stateProvince,individualCount,decimalLatitude,decimalLongitude,eventDate,avg_rad,cf_cvg,NDVI_raw,LandCover_Class,elevation_meters
0,0,Anarhynchus alexandrinus,Mannar,8.0,9.058512,79.855020,2021-01-06,0.66,5.0,3792.0,10,0
1,1,Columba livia,Colombo,31.0,6.927894,79.865005,2024-09-24,25.40,1.0,5692.0,13,12
2,2,Hirundo rustica,Colombo,10.0,6.866285,79.931440,2024-12-23,10.16,7.0,7045.0,13,5


## Step 3: Match each observation to nearest MERRA-2 grid cell by month + location

For each row:
- Extract the year-month from `eventDate`
- Find the nearest lat/lon grid cell in the MERRA-2 data
- Assign the 8 aerosol variable values

In [4]:
# Parse eventDate to extract year-month
df['eventDate'] = pd.to_datetime(df['eventDate'], errors='coerce')
df['year_month'] = df['eventDate'].dt.to_period('M').astype(str)

print(f"Date range in dataset: {df['eventDate'].min()} to {df['eventDate'].max()}")
print(f"Unique months: {df['year_month'].nunique()}")

# Check coverage: which months in the data have matching pollution data?
months_in_data = set(df['year_month'].dropna().unique())
months_available = set(pollution_lookup.keys())
missing_months = months_in_data - months_available
if missing_months:
    print(f'\nWARNING: {len(missing_months)} months in final2.csv have no pollution data:')
    print(sorted(missing_months))
else:
    print('\nAll months in final2.csv have matching pollution data!')

Date range in dataset: 2014-01-01 00:00:00 to 2024-12-31 00:00:00
Unique months: 132

All months in final2.csv have matching pollution data!


In [5]:
# Get the reference grid (same for all files since it's the same spatial subset)
ref_key = list(pollution_lookup.keys())[0]
ref_lat = pollution_lookup[ref_key]['lat']
ref_lon = pollution_lookup[ref_key]['lon']

print(f'Reference grid lat: {ref_lat}')
print(f'Reference grid lon: {ref_lon}')

def find_nearest_idx(array, value):
    """Find index of the nearest value in an array."""
    return np.abs(array - value).argmin()

# Vectorized nearest-neighbor lookup:
# Pre-compute nearest lat/lon indices for all rows at once
lat_values = df['decimalLatitude'].values
lon_values = df['decimalLongitude'].values

# For each observation lat/lon, find the nearest grid index
nearest_lat_idx = np.array([find_nearest_idx(ref_lat, lat) for lat in lat_values])
nearest_lon_idx = np.array([find_nearest_idx(ref_lon, lon) for lon in lon_values])

print(f'\nComputed nearest grid indices for {len(df):,} observations')
print(f'Nearest lat index range: {nearest_lat_idx.min()} to {nearest_lat_idx.max()}')
print(f'Nearest lon index range: {nearest_lon_idx.min()} to {nearest_lon_idx.max()}')

Reference grid lat: [ 6.   6.5  7.   7.5  8.   8.5  9.   9.5 10. ]
Reference grid lon: [79.375 80.    80.625 81.25  81.875]

Computed nearest grid indices for 1,999,208 observations
Nearest lat index range: 0 to 8
Nearest lon index range: 0 to 4


In [6]:
# Initialize new columns with NaN
for var in POLLUTION_VARS:
    df[var] = np.nan

# Process by year-month group for efficiency
year_months = df['year_month'].values
matched = 0
unmatched = 0

for ym in df['year_month'].dropna().unique():
    if ym not in pollution_lookup:
        unmatched += (year_months == ym).sum()
        continue
    
    mask = year_months == ym
    entry = pollution_lookup[ym]
    lat_idx = nearest_lat_idx[mask]
    lon_idx = nearest_lon_idx[mask]
    
    for var in POLLUTION_VARS:
        df.loc[mask, var] = entry[var][lat_idx, lon_idx]
    
    matched += mask.sum()

# Handle rows with NaT eventDate
nat_rows = df['eventDate'].isna().sum()

print(f'Matched: {matched:,} rows')
print(f'Unmatched (missing month): {unmatched:,} rows')
print(f'NaT dates (no match possible): {nat_rows:,} rows')
print(f'\nPollution columns added: {POLLUTION_VARS}')

Matched: 1,999,208 rows
Unmatched (missing month): 0 rows
NaT dates (no match possible): 0 rows

Pollution columns added: ['BCSMASS', 'DUEXTT25', 'DUSMASS', 'OCSMASS', 'SO2SMASS', 'SO4SMASS', 'SSSMASS', 'TOTEXTTAU', 'TOTSCATAU']


## Step 4: Verify and inspect the integrated data

In [7]:
# Drop the helper column
df.drop(columns=['year_month'], inplace=True)

print(f'Final dataset: {len(df):,} rows × {len(df.columns)} columns')
print(f'\nNew columns summary:')
print(df[POLLUTION_VARS].describe().round(6))

# Show a sample
print(f'\nSample rows with pollution data:')
df[['verbatimScientificName', 'stateProvince', 'decimalLatitude', 'decimalLongitude', 
    'eventDate'] + POLLUTION_VARS].head(5)

Final dataset: 1,999,208 rows × 21 columns

New columns summary:
         BCSMASS      DUEXTT25    DUSMASS    OCSMASS   SO2SMASS   SO4SMASS  \
count  1999208.0  1.999208e+06  1999208.0  1999208.0  1999208.0  1999208.0   
mean         0.0  1.144200e-02        0.0        0.0        0.0        0.0   
std          0.0  1.254400e-02        0.0        0.0        0.0        0.0   
min          0.0  6.130000e-04        0.0        0.0        0.0        0.0   
25%          0.0  3.056000e-03        0.0        0.0        0.0        0.0   
50%          0.0  6.857000e-03        0.0        0.0        0.0        0.0   
75%          0.0  1.514900e-02        0.0        0.0        0.0        0.0   
max          0.0  1.248380e-01        0.0        0.0        0.0        0.0   

         SSSMASS     TOTEXTTAU     TOTSCATAU  
count  1999208.0  1.999208e+06  1.999208e+06  
mean         0.0  1.856410e-01  1.761330e-01  
std          0.0  5.275200e-02  5.029800e-02  
min          0.0  7.488600e-02  7.020200e-02

,verbatimScientificName,stateProvince,decimalLatitude,decimalLongitude,eventDate,BCSMASS,DUEXTT25,DUSMASS,OCSMASS,SO2SMASS,SO4SMASS,SSSMASS,TOTEXTTAU,TOTSCATAU
0,Anarhynchus alexandrinus,Mannar,9.058512,79.855020,2021-01-06,5.794512e-10,0.001600,1.548033e-09,2.780749e-09,1.643020e-10,3.372161e-09,3.725563e-08,0.175115,0.165737
1,Columba livia,Colombo,6.927894,79.865005,2024-09-24,3.096564e-10,0.021307,1.578878e-08,1.537093e-09,4.319662e-09,1.169898e-09,5.924096e-08,0.191869,0.184795
2,Hirundo rustica,Colombo,6.866285,79.931440,2024-12-23,9.678071e-10,0.001877,2.117789e-09,4.676763e-09,4.725881e-09,4.424376e-09,3.249729e-08,0.198127,0.187366
3,Geokichla spiloptera,Matale,7.401229,80.690730,2024-09-13,2.822976e-10,0.020004,1.525811e-08,1.535989e-09,2.551352e-09,1.286736e-09,3.795434e-08,0.162928,0.156176
4,Himantopus himantopus,Puttalam,8.154166,79.736084,2024-11-28,8.410959e-10,0.001956,1.985992e-09,4.197754e-09,1.223400e-09,4.910509e-09,4.148581e-08,0.213164,0.203239


## Step 5: Save the integrated dataset

In [8]:
# Save back to final2.csv with the new pollution columns
df.to_csv('final2.csv', index=False)
print(f'Saved updated final2.csv with {len(df.columns)} columns:')
print(list(df.columns))
print(f'\nFile size: {os.path.getsize("final2.csv") / 1e6:.1f} MB')

Saved updated final2.csv with 21 columns:
['index', 'verbatimScientificName', 'stateProvince', 'individualCount', 'decimalLatitude', 'decimalLongitude', 'eventDate', 'avg_rad', 'cf_cvg', 'NDVI_raw', 'LandCover_Class', 'elevation_meters', 'BCSMASS', 'DUEXTT25', 'DUSMASS', 'OCSMASS', 'SO2SMASS', 'SO4SMASS', 'SSSMASS', 'TOTEXTTAU', 'TOTSCATAU']

File size: 593.6 MB
